# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de Gemini.

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [1]:
from google import genai

API_KEY = "[Colocar API_KEY]"

client = genai.Client(api_key=API_KEY)

# Generar contenido con un prompt simple
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hola, ¿Que puedes hacer?"
)

print(response.text)

¡Hola! Como modelo de lenguaje de IA, puedo hacer muchas cosas para ayudarte. Aquí te presento algunas de mis principales capacidades:

*   **Responder preguntas:** Sobre una amplia gama de temas, desde ciencia e historia hasta cultura popular y consejos prácticos.
*   **Generar texto creativo:** Escribir historias, poemas, guiones, canciones, correos electrónicos, cartas, etc.
*   **Resumir información:** Tomar textos largos y extraer los puntos clave de manera concisa.
*   **Traducir idiomas:** Entre varios idiomas (aunque mi fluidez puede variar según el par de idiomas).
*   **Ayudar con la escritura:** Corregir gramática, sugerir mejoras, reescribir pasajes, etc.
*   **Programación y depuración:** Puedo ayudar a escribir código básico, explicar conceptos de programación y depurar errores comunes.
*   **Brainstorming y generación de ideas:** Ayudarte a pensar en ideas para proyectos, nombres, temas, etc.
*   **Conversar:** Mantener una conversación sobre diversos temas de forma cohe

## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [2]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd
import numpy as np

# Cargar el dataset de 20 News Groups
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

# Crear un DataFrame para facilitar el manejo
df = pd.DataFrame({
    'text': newsgroups.data,
    'target': newsgroups.target,
    'target_names': [newsgroups.target_names[i] for i in newsgroups.target]
})

# Mostrar información del dataset
print(f"Total de documentos: {len(df)}")
print(f"Número de categorías: {len(newsgroups.target_names)}")
print("Categorias:")
for category in newsgroups.target_names:
    print(f"- {category}")

Total de documentos: 18846
Número de categorías: 20
Categorias:
- alt.atheism
- comp.graphics
- comp.os.ms-windows.misc
- comp.sys.ibm.pc.hardware
- comp.sys.mac.hardware
- comp.windows.x
- misc.forsale
- rec.autos
- rec.motorcycles
- rec.sport.baseball
- rec.sport.hockey
- sci.crypt
- sci.electronics
- sci.med
- sci.space
- soc.religion.christian
- talk.politics.guns
- talk.politics.mideast
- talk.politics.misc
- talk.religion.misc


### 2.2 Transformo a embeddings

In [ ]:
!pip install -q -U google-genai tenacity

In [3]:
import time
from google.genai import types
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Reintenta automáticamente ante errores transitorios (rate limit, timeouts, etc.)
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=30),
    retry=retry_if_exception_type(Exception)
)
def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    """Obtiene el embedding de un texto usando Gemini (SDK nuevo google-genai)"""
    # Truncar texto si es muy largo (Gemini tiene límites)
    if len(text) > 10000:
        text = text[:10000]

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    # result.embeddings es una lista (uno por cada texto de entrada); tomamos el primero
    return result.embeddings[0].values

In [4]:
def process_documents_in_batches(df, batch_size=10, delay_between_batches=2):
    """Procesa documentos en lotes pequeños"""
    embeddings = []
    total_docs = len(df)

    for i in range(0, total_docs, batch_size):
        batch = df.iloc[i:i+batch_size]
        print(f"Procesando lote {i//batch_size + 1}/{(total_docs + batch_size - 1)//batch_size}")

        for idx, row in batch.iterrows():
            try:
                embedding = get_embedding(row['text'])
            except Exception as e:
                print(f"Error al obtener embedding (doc {idx}): {e}")
                embedding = None
            embeddings.append(embedding)

        # Delay entre lotes (para no pasarte del rate limit gratuito)
        if i + batch_size < total_docs:
            print(f"Esperando {delay_between_batches} segundos antes del siguiente lote...")
            time.sleep(delay_between_batches)

    return embeddings

In [6]:
SAMPLE_SIZE = 20 # Empezar con un número pequeño para probar
df_sample = df.head(SAMPLE_SIZE).copy()

print(f"Generando embeddings para {len(df_sample)} documentos...")
embeddings = process_documents_in_batches(df_sample, batch_size=5, delay_between_batches=2)

df_sample['embedding'] = embeddings
df_sample = df_sample[df_sample['embedding'].notna()]
print(f"Documentos procesados exitosamente: {len(df_sample)}")

Generando embeddings para 20 documentos...
Procesando lote 1/4
Esperando 2 segundos antes del siguiente lote...
Procesando lote 2/4
Esperando 2 segundos antes del siguiente lote...
Procesando lote 3/4
Esperando 2 segundos antes del siguiente lote...
Procesando lote 4/4
Documentos procesados exitosamente: 20


### 2.3 Creo una query y hago la búsqueda

In [7]:
# Crear una query
query = "This is an amazing truck"

# Obtener el embedding de la query (usamos task_type RETRIEVAL_QUERY para queries de búsqueda)
query_embedding = get_embedding(query, task_type="RETRIEVAL_QUERY")
print(f"Embedding de la query creado. Dimensión: {len(query_embedding)}")

Embedding de la query creado. Dimensión: 3072


Obtengo los 5 documentos más similares a mi query

In [8]:
def cosine_similarity(a, b):
    """Calcula la similitud coseno entre dos vectores"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Calcular similitudes
similarities = []
for idx, row in df_sample.iterrows():
    if row['embedding'] is not None:
        sim = cosine_similarity(query_embedding, row['embedding'])
        similarities.append((idx, sim, row['text']))

# Ordenar por similitud
similarities.sort(key=lambda x: x[1], reverse=True)

# Mostrar los 5 más similares
print(f"Los 5 documentos más similares a la query: '{query}'\n")
print("="*80)

for i, (idx, score, text) in enumerate(similarities[:5]):
    print(f"\n{i+1}. Documento ID: {idx}")
    print(f"   Similitud: {score:.4f}")
    print(f"   Categoría: {df_sample.loc[idx, 'target_names']}")
    print(f"   Texto: {text[:300]}...")
    print("-"*80)

Los 5 documentos más similares a la query: 'This is an amazing truck'


1. Documento ID: 0
   Similitud: 0.6274
   Categoría: rec.sport.hockey
   Texto: 

I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
...
--------------------------------------------------------------------------------

2. Documento ID: 8
   Similitud: 0.6233
   Categoría: rec.sport.hockey
   Texto: 


Yeah, it's the second one.  And I believe that price too.  I've been trying
to get a good look at it on the Bruin-Sabre telecasts, and wow! does it ever
look good.  Whoever did that paint job knew what they were doing.  And given
Fuhr's play since he got it, I bet the Bruins are wishing he didn't...
----------------------------------------------------------------------------